### FEATURE ENGINEERING EXPERIMENTATION DECK

In [1]:
import pandas as pd
import numpy as np

In [34]:
df = pd.read_parquet("../data/processed/api_ingest.parquet") 
df.head()

,product_id,product_name,brand,price,product_rating,rating_count,seller_name,stock_qty,category,fetched_time,source
0,5000867,JCO4 LAPTOP BATTERY,HP,24725,0.0,0.0,Konga,1,None,2025-11-11 19:49:08+00:00,api_ingest
1,6566278,Basilisk Gaming Mouse Classic Black,Razer,244000,0.0,0.0,YOMILINCON BRAND,20,None,2025-11-11 19:49:08+00:00,api_ingest
2,5228829,Ht03xl In-built Battery,HP,35075,0.0,0.0,Konga,2,None,2025-11-11 19:49:08+00:00,api_ingest
3,2353031,HP Big-Mouth Laptop Charger - 18.5V 3.5A,HP,10000,4.5,2.0,YOMILINCON BRAND,9,None,2025-11-11 19:49:08+00:00,api_ingest
4,6861874,Baseus Ultrajoy 5w1 5-port Hub docking station...,Baseus,49000,0.0,0.0,YOMILINCON BRAND,2,None,2025-11-11 19:49:08+00:00,api_ingest


In [27]:
df.dtypes

product_id                      int64
product_name                      str
brand                             str
price                           int64
product_rating                float64
rating_count                  float64
seller_name                       str
stock_qty                       int64
category                       object
fetched_time      datetime64[us, UTC]
source                            str
dtype: object

In [28]:
df.drop(columns="category", inplace=True)
df.isna().sum()

product_id         0
product_name       0
brand              0
price              0
product_rating    40
rating_count      40
seller_name        0
stock_qty          0
fetched_time       0
source             0
dtype: int64

In [29]:
# Binning different price tiers into low, mid & high
df["price_tier"] = pd.qcut(df["price"], q=3, labels= ["Low", "Mid", "High"]) 

### BUILDING THE PRODUCT INTELLIGENCE ENGINE

In [30]:
# Checking the distribution of product_id inorder to investigate the distribution of products within the dataset
df["product_id"].value_counts()

product_id
5000867    43
6566278    22
6861874    22
6860173    22
6860155    22
           ..
6986469     1
6986465     1
6986464     1
6986463     1
6986457     1
Name: count, Length: 664, dtype: int64

In [32]:
# Sorting the dataset by fetched_time and product_id to ensure that the latest price and rating information is captured for each product 
df = df.sort_values(["fetched_time", "product_id"]) 
df.head()

,product_id,product_name,brand,price,product_rating,rating_count,seller_name,stock_qty,fetched_time,source,price_tier
3,2353031,HP Big-Mouth Laptop Charger - 18.5V 3.5A,HP,10000,4.5,2.0,YOMILINCON BRAND,9,2025-11-11 19:49:08+00:00,api_ingest,Low
0,5000867,JCO4 LAPTOP BATTERY,HP,24725,0.0,0.0,Konga,1,2025-11-11 19:49:08+00:00,api_ingest,Mid
2,5228829,Ht03xl In-built Battery,HP,35075,0.0,0.0,Konga,2,2025-11-11 19:49:08+00:00,api_ingest,Mid
39,5522604,C270 Hd Webcam - Hd 720p - Widescreen Hd Video,Logitech,80000,0.0,0.0,YOMILINCON BRAND,17,2025-11-11 19:49:08+00:00,api_ingest,Mid
38,5527693,Mini Display Port To Hdmi,Mini,36200,0.0,0.0,YOMILINCON BRAND,17,2025-11-11 19:49:08+00:00,api_ingest,Mid


In [33]:
df_2 = df.groupby("product_id", dropna=False).agg(product_name = ("product_name", "last"), brand = ("brand", "last"), seller_name = ("seller_name", "last"), source = ("source", "last"), first_seen = ("fetched_time", "first"), last_seen = ("fetched_time", "last"),
                                                   observation_count = ("product_id", "size"), avg_price = ("price", "mean"), current_price = ("price", "last"), avg_rating = ("product_rating", "mean"), current_rating = ("product_rating", "last"), 
                                                   total_rating_count = ("product_rating", "sum"), stock_qty = ("stock_qty", "last")).reset_index()
df_2.head() 

,product_id,product_name,brand,seller_name,source,first_seen,last_seen,observation_count,avg_price,current_price,avg_rating,current_rating,total_rating_count,stock_qty
0,2353031,HP Big-Mouth Laptop Charger - 18.5V 3.5A,HP,YOMILINCON BRAND,api_ingest,2025-11-11 19:49:08+00:00,2025-11-11 19:49:08+00:00,1,10000.00000,10000,4.5,4.5,4.5,9
1,4835380,Flexible Usb External Keyboard And Optical Wir...,Havit,YOMILINCON BRAND,api_ingest,2025-12-21 01:41:53+00:00,2026-01-16 01:40:10+00:00,7,10000.00000,10000,0.0,0.0,0.0,15
2,4840254,Optical Usb Mouse,Dell,KACHISON,api_ingest,2026-03-31 02:11:08+00:00,2026-03-31 02:11:08+00:00,1,4000.00000,4000,5.0,5.0,5.0,1
3,4851063,Rgb Backlit Gaming Mouse -black,Havit,YOMILINCON BRAND,api_ingest,2025-12-16 01:28:12+00:00,2025-12-26 01:27:46+00:00,3,19500.00000,19500,4.0,4.0,12.0,7
4,5000867,JCO4 Laptop Battery – High Capacity – Recharge...,HP,Konga,api_ingest,2025-11-11 19:49:08+00:00,2026-05-26 02:57:16+00:00,43,35062.44186,49420,0.0,0.0,0.0,1


### COMPLETED THE PRODUCT FEATURE ENGINEERING NEEDED FOR PRODUCT INTELLIGENCE! 

In [34]:
df_2["price_tier"] = pd.qcut(df_2["current_price"], q=3, labels= ["Low", "Mid", "High"])  

### EXPERIMENTATION COMPLETED - The brand & seller intelligence entails thesame steps which was taken in the product intelligence, Therefore, all i need to do is repurpose the steps and actions carried out in the product intelligence for the rest!! 

In [35]:
df["brand"].value_counts()

brand
Apple       220
Logitech    141
Razer       110
Hp           72
Baseus       67
           ... 
LDNIO         1
2.4g          1
Lingo         1
Pc            1
Jabra         1
Name: count, Length: 169, dtype: int64

In [36]:
df_2 = df.groupby("brand")
df_2["brand"].value_counts()

brand
/           1
1.5m        1
100w        1
10M         2
10meters    2
           ..
Wireless    1
Wiwu        1
X75v2       1
Xiaomi      1
Yvi+        1
Name: count, Length: 169, dtype: int64

In [30]:
import pandas as pd
from pathlib import Path

base = Path("../data/features/api_ingest")

product = pd.read_parquet(base / "product_features.parquet")
brand = pd.read_parquet(base / "brand_features.parquet")
seller = pd.read_parquet(base / "seller_features.parquet")

print("PRODUCT FEATURES")
product.head(50) 

PRODUCT FEATURES


,product_id,product_name,brand,seller_name,source,first_seen,last_seen,days_active,observation_count,avg_price,current_price,price_tier,avg_rating,current_rating,total_rating_count,stock_qty
0,2353031,HP Big-Mouth Laptop Charger - 18.5V 3.5A,HP,YOMILINCON BRAND,api_ingest,2025-11-11 19:49:08+00:00,2025-11-11 19:49:08+00:00,0,1,1.000000e+04,10000,Low,4.500000,4.5,2.0,9
1,4835380,Flexible Usb External Keyboard And Optical Wir...,Havit,YOMILINCON BRAND,api_ingest,2025-12-21 01:41:53+00:00,2026-01-16 01:40:10+00:00,25,7,1.000000e+04,10000,Low,0.000000,0.0,0.0,15
2,4840254,Optical Usb Mouse,Dell,KACHISON,api_ingest,2026-03-31 02:11:08+00:00,2026-03-31 02:11:08+00:00,0,1,4.000000e+03,4000,Low,5.000000,5.0,1.0,1
3,4851063,Rgb Backlit Gaming Mouse -black,Havit,YOMILINCON BRAND,api_ingest,2025-12-16 01:28:12+00:00,2025-12-26 01:27:46+00:00,9,3,1.950000e+04,19500,Mid,4.000000,4.0,3.0,7
4,5000867,JCO4 Laptop Battery – High Capacity – Recharge...,HP,Konga,api_ingest,2025-11-11 19:49:08+00:00,2026-05-26 02:57:16+00:00,195,43,3.506244e+04,49420,High,0.000000,0.0,0.0,1
5,5228829,Ht03xl In-built Battery,HP,Konga,api_ingest,2025-11-11 19:49:08+00:00,2025-11-11 19:49:08+00:00,0,1,3.507500e+04,35075,High,0.000000,0.0,0.0,2
6,5410287,M90 Wired Usb Mouse - Black,Logitech,YOMILINCON BRAND,api_ingest,2025-11-27 04:14:01+00:00,2026-02-06 01:56:35+00:00,70,9,1.200000e+04,12000,Low,1.333333,4.0,3.0,4
7,5517387,87w Usb-c Power Adapter,Apple,YOMILINCON BRAND,api_ingest,2025-11-27 04:14:01+00:00,2026-01-21 01:43:08+00:00,54,7,9.200000e+04,92000,High,0.000000,0.0,0.0,19
8,5520794,4fd Button Usb Wired Mouse,Microsoft,YOMILINCON BRAND,api_ingest,2025-11-27 04:14:01+00:00,2026-02-16 02:01:40+00:00,80,16,3.440000e+04,34400,Mid,0.000000,0.0,0.0,20
9,5522604,C270 Hd Webcam - Hd 720p - Widescreen Hd Video,Logitech,YOMILINCON BRAND,api_ingest,2025-11-11 19:49:08+00:00,2026-02-16 02:01:40+00:00,96,20,8.000000e+04,80000,High,0.000000,0.0,0.0,17


In [21]:
product["product_name"].value_counts().sort_values(ascending=False).head(10)

product_name
Adjustable Laptop And Tablet Stand - Silver                                                   9
Tp-link Cat 6 Lan Cable - 305m                                                                9
Jack Plug To Usb Power Charger Sync Data Transfer Cable For Ipod - 3.5mm                      8
Hdmi Hdtv Extender Amplifier - 8k/4k- 60hz Uhd Hdmi Repeater Female To Female Adapter - 8k    8
Pd100w Usb-c 18.5-20v Small Blue Pin Cable For Hp Laptops                                     8
Hp 150w 19.5v 7.7a For Hp Omen 17- Zbook Charger                                              8
4k Hdmi To Usb 3.0 Video Capture Card                                                         8
Replacement Dell 19.5v3.34a Big Pin Charger And Power Cord                                    8
Portable 3.5mm Desktop Pc Microphone With Stand For Youtube Streaming Recording               7
Type-c Pd Charging Cable To 7.9x5.5mm Laptop Power Adapter For Lenovo Thinkpad T400- 100w     7
Name: count, dtype: int64

In [23]:
product["product_id"].value_counts().head(10) 

product_id
2353031    1
4835380    1
4840254    1
4851063    1
5000867    1
5228829    1
5410287    1
5517387    1
5520794    1
5522604    1
Name: count, dtype: int64

In [31]:
print("\nBRAND FEATURES")
brand["brand"].unique() 


BRAND FEATURES


<ArrowStringArray>
[       '/',     '1.5m',     '100w',      '10M', '10meters',       '11',
  '11-in-1',     '2.4g',   '2.4ghz',       '20',
 ...
      'USB',   'Ugreen',      'Usb',    'Usb-c',     'Wifi', 'Wireless',
     'Wiwu',    'X75v2',   'Xiaomi',     'Yvi+']
Length: 169, dtype: str

In [11]:
print(brand.shape)
print(brand.dtypes)

(169, 13)
brand                       str
product_count             int64
observation_count         int64
seller_count              int64
top_seller                  str
avg_price               float64
median_price            float64
price_tier             category
avg_rating              float64
total_rating_count      float64
rating_coverage_pct     float64
avg_stock_qty           float64
source                      str
dtype: object


In [32]:
print("\nSELLER FEATURES")
print(seller.shape)
print(seller.dtypes)


SELLER FEATURES
(114, 13)
seller_name                 str
product_count             int64
observation_count         int64
brand_count               int64
top_brand                   str
avg_price               float64
median_price            float64
price_tier             category
avg_rating              float64
total_rating_count      float64
rating_coverage_pct     float64
avg_stock_qty           float64
source                      str
dtype: object


In [33]:
seller.head(20)

,seller_name,product_count,observation_count,brand_count,top_brand,avg_price,median_price,price_tier,avg_rating,total_rating_count,rating_coverage_pct,avg_stock_qty,source
0,20percent-cart,4,4,4,Basemo,32750.000000,23000.0,Mid,0.0,0.0,100.000000,20.000000,api_ingest
1,AKAIT COMMUNICATIONS,5,5,1,Microsoft,331500.000000,313500.0,High,0.0,0.0,100.000000,4.600000,api_ingest
2,ARMSTRONG GADGETS STORE,1,1,1,Oraimo,30000.000000,30000.0,Mid,0.0,0.0,100.000000,1.000000,api_ingest
3,Add Moore,41,45,25,Hp,18404.444444,14300.0,Low,0.0,0.0,93.333333,20.000000,api_ingest
4,Alephh Acme Limited,2,2,1,2.4ghz,10000.000000,10000.0,Low,NaN,0.0,0.000000,20.000000,api_ingest
5,Alimosho Store,3,3,2,Basemo,37000.000000,23000.0,Mid,0.0,0.0,100.000000,20.000000,api_ingest
6,All Made,1,1,1,Pd,9000.000000,9000.0,Low,0.0,0.0,100.000000,20.000000,api_ingest
7,Ambless hub,11,11,4,Replacement,11090.909091,11000.0,Low,0.0,0.0,100.000000,20.000000,api_ingest
8,BJ'S SUPERMART,1,2,1,Foldable,45000.000000,45000.0,Mid,0.0,0.0,100.000000,1.000000,api_ingest
9,Big brother,1,1,1,Laptop,15000.000000,15000.0,Low,0.0,0.0,100.000000,1.000000,api_ingest
